# 04 — Partie B2 : dictionnaire de mots-clés seul, sans entraînement

Trois dictionnaires, du plus au moins exposé à nos données annotées :
- **train** : les 200 mots les plus typiques de chaque valeur, extraits des labels du train (`k` choisi sur val en A.4) ;
- **train sans sujets** : le même, moins tous les mots présents dans les conclusions du train (mots de sujet) ;
- **descriptions** : uniquement les mots de `value-categories.json`, sans aucun exemple annoté.

In [1]:
import numpy as np
import pandas as pd
from valeval.data import RAW, load_split, label_matrix, VALUES
from valeval.evaluate import scores, f1_par_valeur, log_result
from valeval.keywords import depuis_descriptions, extraire, mots, predire

train, val = load_split("training"), load_split("validation")
y_va = label_matrix(val)

d_train = extraire(train, 200)
mots_sujets = set().union(*train.conclusion.map(mots))
d_sans_sujets = {v: [m for m in l if m not in mots_sujets] for v, l in d_train.items()}
d_desc = depuis_descriptions(RAW / "value-categories.json")
dicos = {"train": d_train, "train sans sujets": d_sans_sujets, "descriptions": d_desc}

lignes = []
for nom, dico in dicos.items():
    s = scores(y_va, predire(val, dico))
    lignes.append({"dictionnaire": nom, "mots par valeur (moy.)": round(np.mean([len(l) for l in dico.values()])),
                   "précision": s["macro_precision"], "rappel": s["macro_recall"], "macro-F1": s["macro_f1"]})
    log_result(f"B2_dictionnaire_{nom.replace(' ', '_')}", "mots-clés", 0, val=s, temps_s=0, params=0)
pd.DataFrame(lignes).round(3)

,dictionnaire,mots par valeur (moy.),précision,rappel,macro-F1
0,train,200,0.266,0.547,0.334
1,train sans sujets,155,0.275,0.393,0.295
2,descriptions,21,0.323,0.154,0.187


In [2]:
pd.DataFrame({v: (d_desc[v] + [""] * 8)[:8] for v in ["Hedonism", "Tradition", "Humility", "Benevolence: dependability"]})

,Hedonism,Tradition,Humility,Benevolence: dependability
0,enjoyable,adequate,accept,backing
1,gratification,certain,acceptance,clear
2,leisure,customs,accepted,confident
3,pleasure,devote,achieved,dependability
4,sensuous,faith,arrogance,dependable
5,,family's,brag,foster
6,,holding,circumstances,friend
7,,honoring,demoting,friends


### F1 par valeur

In [3]:
pd.DataFrame({nom: pd.Series(f1_par_valeur(y_va, predire(val, dico))) for nom, dico in dicos.items()}).assign(
    toujours_oui=pd.Series(f1_par_valeur(y_va, np.ones_like(y_va)))).sort_values("descriptions", ascending=False).round(3)

,train,train sans sujets,descriptions,toujours_oui
Achievement,0.540,0.427,0.381,0.465
Benevolence: caring,0.523,0.410,0.351,0.501
Self-direction: action,0.472,0.390,0.320,0.415
Power: resources,0.325,0.295,0.274,0.130
Universalism: nature,0.346,0.338,0.254,0.126
Universalism: objectivity,0.371,0.352,0.248,0.327
Tradition,0.261,0.272,0.246,0.166
Universalism: concern,0.573,0.548,0.245,0.532
Security: societal,0.507,0.420,0.241,0.409
Security: personal,0.650,0.487,0.232,0.572


### Contamination : le test partage-t-il des mots avec les dictionnaires ?
On vérifie **sans regarder les labels du test** quelle part des mots du dictionnaire train apparaît dans les textes de val et du test. Le dictionnaire n'a vu ni l'un ni l'autre.

In [4]:
test_txt = load_split("test")
vocab = {"val": set().union(*(val.conclusion + " " + val.premise).map(mots)),
         "test": set().union(*(test_txt.conclusion + " " + test_txt.premise).map(mots))}
tous = set(m for l in d_train.values() for m in l)
print({nom: f"{len(tous & v) / len(tous):.1%} des mots du dictionnaire train y apparaissent" for nom, v in vocab.items()})
print("Argument ids communs train / val / test :", len(set(train.id) & set(val.id)), len(set(train.id) & set(test_txt.id)))

{'val': '79.0% des mots du dictionnaire train y apparaissent', 'test': '74.5% des mots du dictionnaire train y apparaissent'}
Argument ids communs train / val / test : 0 0


**Lecture (Q5) :**

| Dictionnaire | Précision | Rappel | Macro-F1 val |
|---|---|---|---|
| train (`k` = 200) | 0,266 | 0,547 | **0,334** |
| train sans sujets | 0,275 | 0,393 | 0,295 |
| descriptions seules | 0,323 | 0,154 | 0,187 |

- **Ce que cela révèle :** les mots écrits pour *décrire* une valeur apparaissent rarement dans les arguments (rappel 0,15) : **la valeur est rarement nommée**. Le dictionnaire issu des descriptions fait moins bien que répondre « oui » partout (0,272).
- **Mots du train :** ils marchent mieux, car ils captent le vocabulaire réel des arguments, et non la définition de la valeur.
- **Mots de sujet :** les retirer fait perdre 0,039. C'est un majorant de leur apport, car cette suppression enlève aussi des mots généraux présents dans les conclusions (par exemple *health*).

**Contamination :**
- Dictionnaire **train** : pas vraiment « sans entraînement », il dépend des labels du train. Son `k` a été choisi sur val, donc son score val est légèrement optimiste. Il n'a vu ni les textes ni les labels du test, et aucun `id` ni aucune conclusion n'est commun entre les splits.
- Dictionnaire **descriptions** : aucun exemple annoté utilisé par nous. On ne peut pas garantir que les auteurs du fichier n'ont pas été influencés par le corpus en rédigeant leurs exemples.
- 74,5 % des mots du dictionnaire train apparaissent dans les textes du test : ce sont surtout des mots courants, ce n'est **pas** en soi une fuite.